In [1]:
# 导入所有必需的库
import torch
import torch.nn as nn
import torch.nn.functional as F
import einops

# 01. RMSNorm Tutorial | 均方根层归一化 (RMSNorm)

### LayerNorm 核心公式与张量维度

给定输入向量 $x \in \mathbb{R}^d$，LayerNorm 的输出 $y$ 为：

1. **计算均值与方差：**
   $$ \mu = \frac{1}{d} \sum_{i=1}^d x_i $$
   $$ \sigma^2 = \frac{1}{d} \sum_{i=1}^d (x_i - \mu)^2 + \epsilon $$
   其中 $\epsilon$ 是为了防止除以 0 的极小值（如 `1e-5`）。

2. **归一化：**
   $$ \hat{x} = \frac{x - \mu}{\sqrt{\sigma^2}} $$

3. **缩放并平移：**
   $$ y = \hat{x} \odot \gamma + \beta $$
   其中 $\gamma \in \mathbb{R}^d$ 是可学习的缩放权重（Weight），$\beta \in \mathbb{R}^d$ 是可学习的偏移参数（Bias）。

**与 RMSNorm 的对比：**
- LayerNorm 包含**均值中心化**（减去 $\mu$）和**方差缩放**两步；RMSNorm 只做 RMS 缩放，无中心化。
- LayerNorm 有**两个**可学习参数（$\gamma$ 和 $\beta$），RMSNorm 仅有 $\gamma$。
- LayerNorm 是原始 Transformer 的标准归一化，RMSNorm 是其变体，计算量更小。

### RMSNorm 核心公式与张量维度

给定输入向量 $x \in \mathbb{R}^d$，RMSNorm 的输出 $y$ 为：

1. **计算均方根 (RMS)：**
   $$ \text{RMS}(x) = \sqrt{\frac{1}{d} \sum_{i=1}^d x_i^2 + \epsilon} $$
   其中 $\epsilon$ 是为了防止除以 0 的极小值（如 `1e-6`）。

2. **归一化并缩放 (Scale)：**
   $$ y = \frac{x}{\text{RMS}(x)} \odot \gamma $$
   其中 $\gamma \in \mathbb{R}^d$ 是可学习的权重参数（Weight）。**RMSNorm 没有偏置项 (Bias)**。


### 数值溢出 (Numerical Overflow)

> **工程经验：为什么要强制转换精度？**
> 现代大模型训练与推理几乎都会使用混合精度 (AMP) 或半精度格式 (`FP16`) 以节省显存。但在执行平方和均值操作前，通常需要显式地将其转换为 `float32` 计算。待归一化计算完毕后，再将结果转换回原有精度。

In [2]:
class RMSNorm(nn.Module):
    def __init__(self, hidden_size: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        # ==========================================
        # TODO 1: 定义可学习参数 weight，并初始化为全 1
        # 形状: [hidden_size]
        # 提示: 使用 nn.Parameter 包装张量使其可学习
        # ==========================================
        self.weight = nn.Parameter(torch.ones(hidden_size))
        pass
        

    def _norm(self, x: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # TODO 2: 实现 RMSNorm 核心计算逻辑
        # 提示: 
        # 1. 为防止 FP16 溢出，需要在高精度下计算
        # 2. 计算输入的均方值（平方后求均值），注意保持维度以便广播
        # 3. 使用均方根的倒数进行归一化，torch.rsqrt 比 1/sqrt 更快
        # 4. 返回归一化后的结果（保持高精度，便于后续操作）
        # ==========================================
        x_fp32 = x if x.dtype == torch.float32 else x.float()
        variance = x_fp32.pow(2).mean(dim=-1, keepdim=True)
        return x_fp32 * torch.rsqrt(variance + self.eps)
        pass


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ==========================================
        # TODO 3: 组合归一化与权重缩放
        # 提示: 调用 _norm 进行归一化，乘以可学习的 weight，最后转回输入精度
        # ==========================================
        weight = self.weight.to(x.dtype)
        return (weight * self._norm(x)).to(x.dtype)
        pass